# 🔐 MojoPQC-SCA: Neural Side-Channel Analysis for Post-Quantum Cryptography

**End-to-End Training Pipeline on Google Colab GPU**

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/hardikxk/pqc-sca/blob/main/notebooks/colab_train.ipynb)

---

This notebook runs the **complete MojoPQC-SCA training pipeline** on a free Colab GPU runtime:

| Step | Description | GPU Required |
|------|-------------|--------------|
| 1 | Generate synthetic ML-KEM-512 side-channel traces | ❌ |
| 2 | Stream-preprocess raw HDF5 → filtered traces | ❌ |
| 3 | Train baseline 1D-CNN model | ✅ |
| 4 | Train CNN + MPS tensor-network model | ✅ |
| 5 | Evaluate guessing entropy (GE) curves | ❌ |
| 6 | Export to ONNX + Int8 quantization | ❌ |
| 7 | Benchmark & compare both models | ❌ |

> **Runtime**: Select **Runtime → Change runtime type → T4 GPU** before running.
>
> Estimated wall-clock time: **~10–15 minutes** on a free T4.

## 📦 Step 0: Environment Setup

Install all dependencies and clone the repository. This cell handles:
- Python packages (PyTorch, h5py, ONNX Runtime, etc.)
- Repository cloning from GitHub
- GPU availability verification

In [ ]:
# ── Install dependencies ────────────────────────────────────────────
!pip install -q numpy scipy h5py pyyaml psutil matplotlib torch torchvision tqdm onnx onnxruntime

# ── Clone or update the repository ─────────────────────────────────
import os
REPO_DIR = '/content/MojoPQC-SCA'

if os.path.exists(REPO_DIR):
    print(f'Repository already exists at {REPO_DIR}, pulling latest...')
    !cd {REPO_DIR} && git pull
else:
    !git clone https://github.com/hardikxk/pqc-sca.git {REPO_DIR}

os.chdir(REPO_DIR)
print(f'\n✅ Working directory: {os.getcwd()}')

In [ ]:
# ── Verify GPU runtime ─────────────────────────────────────────────
import torch
import platform

print('=' * 60)
print('🖥️  SYSTEM INFORMATION')
print('=' * 60)
print(f'Python     : {platform.python_version()}')
print(f'PyTorch    : {torch.__version__}')
print(f'CUDA avail : {torch.cuda.is_available()}')

if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    gpu_mem = torch.cuda.get_device_properties(0).total_mem / (1024**3)
    print(f'GPU        : {gpu_name}')
    print(f'GPU Memory : {gpu_mem:.1f} GB')
    DEVICE = 'cuda'
else:
    print('⚠️  No GPU detected. Training will use CPU (much slower).')
    print('   Go to Runtime → Change runtime type → T4 GPU')
    DEVICE = 'cpu'

print(f'\nSelected device: {DEVICE}')
print('=' * 60)

---

## 🧪 Step 1: Generate Synthetic ML-KEM-512 Side-Channel Traces

We generate realistic synthetic power traces that simulate the **NTT butterfly leakage** pattern found in ML-KEM-512 (formerly CRYSTALS-Kyber) running on ARM Cortex-M4.

The synthetic generator models:
- **Hamming-weight leakage**: Each trace encodes `HW(S-box[key ⊕ plaintext])` at NTT butterfly positions
- **Gaussian noise**: σ = 1.0 (configurable) to simulate oscilloscope noise
- **Temporal jitter**: ±50 sample random misalignment (realistic for EM probes)
- **HDF5 chunked storage**: Streams data in chunks of 256 traces for memory efficiency

| Parameter | Value |
|-----------|-------|
| Profiling traces | 10,000 |
| Attack traces | 2,000 |
| Raw trace length | 100,000 samples |
| Target classes | 256 (one per byte value) |

In [ ]:
# ── Generate synthetic dataset ─────────────────────────────────────
!python scripts/01_generate_synthetic_dataset.py --config config/colab.yaml

# ── Inspect the raw dataset ────────────────────────────────────────
import h5py
import numpy as np

with h5py.File('data/raw/synthetic_pqc.h5', 'r') as f:
    prof_shape = f['traces/profiling'].shape
    atk_shape  = f['traces/attack'].shape
    print(f'\n📊 Raw Dataset Summary:')
    print(f'   Profiling traces : {prof_shape[0]:,} × {prof_shape[1]:,} samples')
    print(f'   Attack traces    : {atk_shape[0]:,} × {atk_shape[1]:,} samples')
    print(f'   Storage size     : {os.path.getsize("data/raw/synthetic_pqc.h5") / 1e6:.1f} MB')
    print(f'   Label range      : [{f["labels/profiling"][:].min()}, {f["labels/profiling"][:].max()}]')

In [ ]:
# ── Visualize sample raw traces ────────────────────────────────────
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['figure.dpi'] = 120
matplotlib.rcParams['font.size'] = 10

with h5py.File('data/raw/synthetic_pqc.h5', 'r') as f:
    traces = f['traces/profiling'][:5]  # first 5 traces
    labels = f['labels/profiling'][:5]

fig, axes = plt.subplots(2, 1, figsize=(14, 6), gridspec_kw={'height_ratios': [3, 1]})

# Trace overlay
ax = axes[0]
colors = plt.cm.viridis(np.linspace(0, 1, 5))
for i in range(5):
    ax.plot(traces[i, ::50], alpha=0.7, linewidth=0.5, color=colors[i],
            label=f'Trace {i} (label={labels[i]})')
ax.set_title('Raw Power Traces — ML-KEM-512 NTT Butterfly Leakage', fontweight='bold')
ax.set_ylabel('Amplitude (a.u.)')
ax.legend(fontsize=8, loc='upper right')
ax.grid(alpha=0.2)

# Mean + std
ax2 = axes[1]
with h5py.File('data/raw/synthetic_pqc.h5', 'r') as f:
    batch = f['traces/profiling'][:100, ::50]
mean_trace = batch.mean(axis=0)
std_trace = batch.std(axis=0)
x = np.arange(len(mean_trace))
ax2.plot(x, mean_trace, color='#2196F3', linewidth=1)
ax2.fill_between(x, mean_trace - std_trace, mean_trace + std_trace,
                 alpha=0.2, color='#2196F3')
ax2.set_xlabel('Sample index (downsampled 50×)')
ax2.set_ylabel('Mean ± σ')
ax2.set_title('Signal-to-Noise Profile (100-trace average)', fontsize=10)
ax2.grid(alpha=0.2)

plt.tight_layout()
plt.savefig('results/figures/raw_traces_colab.png', dpi=160, bbox_inches='tight')
plt.show()
print('✅ Raw trace visualization saved.')

---

## 🔧 Step 2: Streaming Preprocessing

The preprocessing pipeline converts 100,000-sample raw traces into **5,000-sample normalized traces** through:

1. **Band-pass filtering** — 11-tap kernel to remove DC offset and high-frequency noise
2. **Temporal alignment** — Cross-correlation-based jitter correction (±500 sample shift)
3. **Dimensionality reduction** — 20× downsampling via strided selection
4. **Z-score normalization** — Per-trace zero-mean, unit-variance standardization

All operations run in **constant-memory streaming mode** (chunk_size=256 traces at a time).

In [ ]:
# ── Run streaming preprocessing ───────────────────────────────────
!python scripts/02_python_preprocess_baseline.py --config config/colab.yaml

# ── Validate processed dataset ────────────────────────────────────
!python scripts/12_validate_dataset.py --input data/processed/python_processed.h5 --strict

with h5py.File('data/processed/python_processed.h5', 'r') as f:
    prof = f['traces/profiling']
    print(f'\n📐 Processed Dataset:')
    print(f'   Shape     : {prof.shape[0]:,} traces × {prof.shape[1]:,} samples')
    print(f'   Reduction : {100_000} → {prof.shape[1]} ({100_000 // prof.shape[1]}× compression)')
    print(f'   dtype     : {prof.dtype}')
    print(f'   File size : {os.path.getsize("data/processed/python_processed.h5") / 1e6:.1f} MB')

In [ ]:
# ── Visualize pre vs post processing ──────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

with h5py.File('data/raw/synthetic_pqc.h5', 'r') as f:
    raw = f['traces/profiling'][0]
with h5py.File('data/processed/python_processed.h5', 'r') as f:
    proc = f['traces/profiling'][0]

axes[0].plot(raw[::20], linewidth=0.5, color='#FF5722')
axes[0].set_title('Before: Raw Trace (100K samples)', fontweight='bold')
axes[0].set_ylabel('Amplitude')
axes[0].grid(alpha=0.2)

axes[1].plot(proc, linewidth=0.5, color='#4CAF50')
axes[1].set_title('After: Processed Trace (5K samples)', fontweight='bold')
axes[1].set_ylabel('Normalized amplitude')
axes[1].grid(alpha=0.2)

plt.tight_layout()
plt.savefig('results/figures/preprocessing_comparison.png', dpi=160, bbox_inches='tight')
plt.show()
print('✅ Preprocessing comparison saved.')

---

## 🧠 Step 3: Model Architecture

We train two models and compare them:

### Model A: Baseline 1D-CNN
```
Input (1, 5000) → Conv1D(1→16, k=11) → BN → ReLU → AvgPool(10)
                 → Conv1D(16→32, k=11) → BN → ReLU → AvgPool(10)
                 → AdaptiveAvgPool(1) → Linear(32→256)
```

### Model B: CNN + MPS Tensor Network (our contribution)
```
Same CNN backbone → AdaptiveAvgPool(1)
                  → MPSLayer(32→256, bond_dim=8)
                      ├── SiteEncoder: Linear(32 → 2×5 sites)
                      ├── 5 TN Cores: (8, 2, 8) tensors
                      └── Output: Linear(8→256)
```

The MPS head replaces the dense `Linear(32→256)` layer with a **tensor-network contraction** that captures inter-feature correlations with far fewer parameters (~9,226 vs ~200K budget).

In [ ]:
# ── Inspect model architectures ───────────────────────────────────
import sys
sys.path.insert(0, '.')

from mojopqc_sca.models.cnn import LightweightCNN, count_parameters
from mojopqc_sca.models.cnn_mps import CNNWithMPS

cnn = LightweightCNN(5000, 256)
mps = CNNWithMPS(5000, 256, bond_dim=8)

print('=' * 60)
print('📊 MODEL PARAMETER COMPARISON')
print('=' * 60)
cnn_params = count_parameters(cnn)
mps_params = count_parameters(mps)
print(f'Baseline CNN     : {cnn_params:,} parameters')
print(f'CNN + MPS (ours) : {mps_params:,} parameters')
print(f'Budget limit     : 200,000 parameters')
print(f'Budget usage     : {mps_params / 200_000 * 100:.1f}%')
print(f'Compression      : {cnn_params / mps_params:.1f}× fewer vs baseline')
print('=' * 60)

# Detailed layer breakdown
print('\n🔍 CNN+MPS Layer Breakdown:')
for name, param in mps.named_parameters():
    print(f'   {name:40s} → {param.numel():>6,} params  {list(param.shape)}')
print(f'   {"TOTAL":40s} → {mps_params:>6,} params')

---

## 🏋️ Step 4: Training

We train both models for **30 epochs** with:
- **Optimizer**: Adam (lr = 1e-3)
- **Loss**: Cross-entropy (256 classes)
- **Validation split**: 80/20 stratified random
- **Batch size**: 128
- **Device**: Auto-selects CUDA if available

Training logs (loss, accuracy, timing) are saved as CSV files for later analysis.

In [ ]:
# ── Train baseline CNN ────────────────────────────────────────────
print('🏋️  Training Baseline CNN...')
print('=' * 60)
!python scripts/04_train_cnn.py \
    --input data/processed/python_processed.h5 \
    --device {DEVICE} \
    --epochs 30 \
    --batch-size 128
print('\n✅ Baseline CNN training complete.')

In [ ]:
# ── Train CNN + MPS tensor network ────────────────────────────────
print('🏋️  Training CNN + MPS Tensor Network...')
print('=' * 60)
!python scripts/05_train_cnn_mps.py \
    --input data/processed/python_processed.h5 \
    --device {DEVICE} \
    --epochs 30 \
    --batch-size 128 \
    --bond-dim 8
print('\n✅ CNN+MPS training complete.')

In [ ]:
# ── Plot training curves ──────────────────────────────────────────
import pandas as pd

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

models_data = {}
for name, path, color in [
    ('CNN (baseline)', 'results/logs/cnn_training.csv', '#FF5722'),
    ('CNN+MPS (ours)', 'results/logs/cnn_mps_training.csv', '#2196F3'),
]:
    if os.path.exists(path):
        df = pd.read_csv(path)
        models_data[name] = df

        # Training Loss
        axes[0].plot(df['epoch'], df['train_loss'], label=name,
                     color=color, linewidth=2)
        # Training Accuracy
        axes[1].plot(df['epoch'], df['train_accuracy'] * 100, label=name,
                     color=color, linewidth=2, linestyle='--')
        # Validation Accuracy
        axes[1].plot(df['epoch'], df['validation_accuracy'] * 100, label=f'{name} (val)',
                     color=color, linewidth=2)
        # Epoch time
        axes[2].plot(df['epoch'], df['epoch_time_seconds'], label=name,
                     color=color, linewidth=2)

axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Cross-Entropy Loss')
axes[0].set_title('Training Loss', fontweight='bold'); axes[0].legend(); axes[0].grid(alpha=0.2)

axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Accuracy (%)')
axes[1].set_title('Accuracy (train + validation)', fontweight='bold'); axes[1].legend(fontsize=8); axes[1].grid(alpha=0.2)

axes[2].set_xlabel('Epoch'); axes[2].set_ylabel('Time (seconds)')
axes[2].set_title('Epoch Training Time', fontweight='bold'); axes[2].legend(); axes[2].grid(alpha=0.2)

plt.suptitle(f'Training Curves — Device: {DEVICE.upper()}', fontweight='bold', fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig('results/figures/training_curves_colab.png', dpi=160, bbox_inches='tight')
plt.show()

# Print summary table
print('\n📊 Final Training Results:')
print(f'{"Model":<20s} {"Val Acc":>10s} {"Final Loss":>12s} {"Epoch Time":>12s}')
print('-' * 56)
for name, df in models_data.items():
    last = df.iloc[-1]
    print(f'{name:<20s} {last["validation_accuracy"]*100:>9.2f}% {last["train_loss"]:>12.4f} {last["epoch_time_seconds"]:>10.3f}s')

---

## 📈 Step 5: Guessing Entropy Evaluation

**Guessing entropy (GE)** is the gold-standard metric for side-channel attacks. It measures how many attack traces are needed to reduce the key-candidate rank to 1 (i.e., recover the correct key).

- **GE = 1** means the correct key is the top-ranked candidate → **full key recovery**
- Lower GE with fewer traces = more efficient attack = better model

In [ ]:
# ── Evaluate guessing entropy ─────────────────────────────────────
!python scripts/06_evaluate_ge.py \
    --input data/processed/python_processed.h5 \
    --checkpoint results/models/cnn_mps.pt \
    --step 50

In [ ]:
# ── Plot guessing entropy curve ───────────────────────────────────
import json

ge_path = 'results/benchmarks/ge_results.json'
if os.path.exists(ge_path):
    ge_data = json.loads(open(ge_path).read())
    traces_n = [int(k) for k in ge_data.keys()]
    ge_vals  = list(ge_data.values())

    fig, ax = plt.subplots(figsize=(10, 5))
    ax.plot(traces_n, ge_vals, 'o-', color='#E91E63', linewidth=2,
            markersize=4, markerfacecolor='white', markeredgewidth=1.5)
    ax.axhline(y=1, color='#4CAF50', linestyle='--', linewidth=1.5,
               label='Full key recovery (GE=1)')
    ax.fill_between(traces_n, ge_vals, alpha=0.1, color='#E91E63')
    ax.set_xlabel('Number of Attack Traces', fontsize=12)
    ax.set_ylabel('Guessing Entropy (key rank)', fontsize=12)
    ax.set_title('Guessing Entropy Curve — CNN+MPS Model', fontweight='bold', fontsize=14)
    ax.set_yscale('log')
    ax.legend(fontsize=11)
    ax.grid(alpha=0.2)
    plt.tight_layout()
    plt.savefig('results/figures/ge_curve_colab.png', dpi=160, bbox_inches='tight')
    plt.show()

    # Key recovery milestone
    for n, ge in zip(traces_n, ge_vals):
        if ge <= 1.0:
            print(f'🔑 Full key recovery achieved at {n} traces!')
            break
    else:
        print(f'🔑 Final GE at {traces_n[-1]} traces: {ge_vals[-1]:.2f}')
else:
    print('⚠️  GE results not found. Run evaluation first.')

---

## 🏆 Step 6: Model Comparison & Benchmarks

Head-to-head comparison of the baseline CNN versus our CNN+MPS model across:
- Parameter count
- Validation accuracy
- Attack efficiency (GE at various trace counts)
- Inference speed
- Memory footprint

In [ ]:
# ── Run full benchmark suite ──────────────────────────────────────
!python scripts/07_benchmark_models.py \
    --input data/processed/python_processed.h5

In [ ]:
# ── Display comparison table ──────────────────────────────────────
comparison_path = 'results/benchmarks/model_comparison.json'
if os.path.exists(comparison_path):
    data = json.loads(open(comparison_path).read())

    print('\n' + '=' * 72)
    print('🏆 MODEL COMPARISON: CNN vs CNN+MPS')
    print('=' * 72)
    print(f'{"Metric":<35s} {"CNN (baseline)":>17s} {"CNN+MPS (ours)":>17s}')
    print('-' * 72)

    cnn_d = data[0]; mps_d = data[1]

    metrics = [
        ('Parameters', f"{cnn_d['parameters']:,}", f"{mps_d['parameters']:,}"),
        ('Val. Accuracy', f"{(cnn_d['validation_accuracy'] or 0)*100:.2f}%", f"{(mps_d['validation_accuracy'] or 0)*100:.2f}%"),
        ('GE @ 100 traces', f"{cnn_d.get('ge_100', 'N/A')}", f"{mps_d.get('ge_100', 'N/A')}"),
        ('GE @ 500 traces', f"{cnn_d.get('ge_500', 'N/A')}", f"{mps_d.get('ge_500', 'N/A')}"),
        ('GE @ all traces', f"{cnn_d['ge_all']:.2f}", f"{mps_d['ge_all']:.2f}"),
        ('Inference (sec)', f"{cnn_d['inference_seconds']:.3f}", f"{mps_d['inference_seconds']:.3f}"),
        ('Peak memory (MiB)', f"{cnn_d['peak_memory_bytes']/1e6:.1f}", f"{mps_d['peak_memory_bytes']/1e6:.1f}"),
    ]
    for metric, v1, v2 in metrics:
        print(f'{metric:<35s} {v1:>17s} {v2:>17s}')
    print('=' * 72)

    # Bar chart comparison
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))

    names = ['CNN', 'CNN+MPS']
    colors = ['#FF5722', '#2196F3']

    axes[0].bar(names, [cnn_d['parameters'], mps_d['parameters']], color=colors)
    axes[0].set_title('Parameters', fontweight='bold')
    axes[0].set_ylabel('Count')

    axes[1].bar(names, [cnn_d['ge_all'], mps_d['ge_all']], color=colors)
    axes[1].set_title('GE @ all traces (lower = better)', fontweight='bold')
    axes[1].set_ylabel('Guessing Entropy')

    axes[2].bar(names, [(cnn_d['validation_accuracy'] or 0)*100,
                        (mps_d['validation_accuracy'] or 0)*100], color=colors)
    axes[2].set_title('Validation Accuracy', fontweight='bold')
    axes[2].set_ylabel('Accuracy (%)')

    plt.suptitle('CNN vs CNN+MPS Comparison', fontweight='bold', fontsize=13, y=1.02)
    plt.tight_layout()
    plt.savefig('results/figures/model_comparison_colab.png', dpi=160, bbox_inches='tight')
    plt.show()
else:
    print('⚠️  Run benchmark script first.')

---

## 📦 Step 7: ONNX Export & Int8 Quantization

Export the trained CNN+MPS model for **edge deployment**:
- **FP32 ONNX** — Standard export for GPU/CPU inference
- **Int8 ONNX** — Dynamic quantization for embedded/Cortex-M deployment

The Int8 model reduces the file size by ~4× while maintaining comparable attack accuracy.

In [ ]:
# ── Export to ONNX ────────────────────────────────────────────────
!python scripts/09_export_onnx.py

# ── Compare model sizes ──────────────────────────────────────────
from pathlib import Path

model_files = [
    ('PyTorch checkpoint (.pt)', 'results/models/cnn_mps.pt'),
    ('ONNX FP32 (.onnx)', 'results/models/cnn_mps.onnx'),
    ('ONNX Int8 (.onnx)', 'results/models/cnn_mps_int8.onnx'),
]

print('\n📦 Exported Model Sizes:')
print(f'{"Format":<35s} {"Size":>10s}')
print('-' * 47)
for label, path in model_files:
    if Path(path).exists():
        size = Path(path).stat().st_size
        print(f'{label:<35s} {size/1024:>8.1f} KB')
    else:
        print(f'{label:<35s} {"N/A":>10s}')

In [ ]:
# ── Run ONNX inference benchmark ─────────────────────────────────
!python scripts/10_local_inference.py \
    --model results/models/cnn_mps_int8.onnx \
    --input data/processed/python_processed.h5

inf_path = 'results/benchmarks/local_inference.json'
if os.path.exists(inf_path):
    inf_data = json.loads(open(inf_path).read())
    print(f'\n⚡ ONNX Int8 Inference Results:')
    print(f'   Speed     : {inf_data["milliseconds_per_trace"]:.4f} ms/trace')
    print(f'   Throughput: {1000/inf_data["milliseconds_per_trace"]:.0f} traces/sec')
    print(f'   GE (all)  : {inf_data["ge_all"]:.2f}')
    print(f'   Memory    : {inf_data["peak_memory_bytes"]/1e6:.1f} MB')

---

## 🎯 Summary & Key Findings

This notebook demonstrated the complete MojoPQC-SCA pipeline. Key takeaways:

1. **MPS tensor networks** provide a parameter-efficient alternative to dense classifiers for side-channel analysis
2. The CNN+MPS model uses only **~9,226 parameters** (4.6% of the 200K budget)
3. **Guessing entropy converges** within the available attack traces, demonstrating viable key recovery
4. **Int8 quantization** reduces model size by ~4× with negligible accuracy loss
5. The entire pipeline runs on a **free Colab T4 GPU** in ~15 minutes

In [ ]:
# ── Download artifacts ────────────────────────────────────────────
# Uncomment the lines below to download key artifacts to your machine.

from google.colab import files
import shutil

# Create a zip of all results
shutil.make_archive('/content/mojopqc_results', 'zip', 'results')
print('📥 Downloading results archive...')
files.download('/content/mojopqc_results.zip')

# Or download individual files:
# files.download('results/models/cnn_mps.pt')
# files.download('results/models/cnn_mps_int8.onnx')
# files.download('results/figures/ge_curve.png')
# files.download('results/figures/training_curves_colab.png')